# User Behaviour Feature Engineering

Creates the user-level dataset used by the User Behaviour dashboard. Pareto calculations and duplicate exploratory cells were removed because they were not used in the final report.

## 1. Load and validate the user-level dataset

Each row represents one user. The notebook creates Power BI-ready behavioural features and distribution bins.

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("../data/processed")
INPUT_FILE = DATA_DIR / "user_features.csv"
OUTPUT_FILE = DATA_DIR / "user_features_powerbi.csv"

user_df = pd.read_csv(INPUT_FILE)
print(f"Rows: {len(user_df):,}")
user_df.head()


Rows: 59,232


,uid,total_views,unique_videos,unique_authors,unique_musics,finish_count,like_count,avg_duration_time
0,0,34,34,31,31,18,0,12.06
1,1,28,28,28,26,14,1,12.36
2,2,56,56,56,47,19,0,10.36
3,3,117,117,116,89,60,1,9.98
4,4,123,123,117,94,77,0,10.85


In [5]:
required_columns = [
    "uid", "total_views", "unique_videos", "unique_authors",
    "unique_musics", "finish_count", "like_count", "avg_duration_time"
]

missing_columns = [col for col in required_columns if col not in user_df.columns]
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

if not user_df["uid"].is_unique:
    raise ValueError("The input must contain one row per user.")


## 2. Create behavioural rates

Rates are stored as decimals between 0 and 1 and should be formatted as percentages in Power BI.

In [6]:
user_df["finish_rate"] = np.where(
    user_df["total_views"] > 0,
    user_df["finish_count"] / user_df["total_views"],
    0
)

user_df["like_rate"] = np.where(
    user_df["total_views"] > 0,
    user_df["like_count"] / user_df["total_views"],
    0
)

user_df["avg_duration_time"] = user_df["avg_duration_time"].round(2)
user_df["finish_rate"] = user_df["finish_rate"].round(4)
user_df["like_rate"] = user_df["like_rate"].round(4)


## 3. Createdistribution bins

Numeric order columns are included so Power BI can sort labels correctly.

In [7]:
def add_ordered_bin(data, source, bins, labels, output):
    data[output] = pd.cut(
        data[source],
        bins=bins,
        labels=labels,
        right=True,
        include_lowest=True
    )
    order_map = {label: order for order, label in enumerate(labels, start=1)}
    data[f"{output}_order"] = (
        data[output].astype("string").map(order_map).astype("Int64")
    )

add_ordered_bin(
    user_df, "total_views",
    [0, 5, 10, 25, 50, 100, np.inf],
    ["1–5", "6–10", "11–25", "26–50", "51–100", "101+"],
    "view_bin"
)

add_ordered_bin(
    user_df, "like_count",
    [-1, 0, 1, np.inf],
    ["0 likes", "1 like", "2+ likes"],
    "like_bin"
)

add_ordered_bin(
    user_df, "finish_count",
    [-1, 0, 5, 10, 20, 30, 50, np.inf],
    ["0 finishes", "1–5 finishes", "6–10 finishes",
     "11–20 finishes", "21–30 finishes", "31–50 finishes", "51+ finishes"],
    "finish_bin"
)

add_ordered_bin(
    user_df, "avg_duration_time",
    [1, 10, 12, 15, np.inf],
    ["≤10s", "10–12s", "12–15s", "15s+"],
    "duration_bin"
)


## 4. Validate and export

In [8]:
bin_columns = ["view_bin", "like_bin", "finish_bin", "duration_bin"]
order_columns = [f"{column}_order" for column in bin_columns]

for column in bin_columns:
    user_df[column] = user_df[column].astype("string")

assert user_df[bin_columns].notna().all().all(), "A user was not assigned to a bin."
assert user_df[order_columns].notna().all().all(), "A bin order is missing."
assert user_df["finish_rate"].between(0, 1).all()
assert user_df["like_rate"].between(0, 1).all()

powerbi_columns = [
    "uid", "total_views", "unique_videos", "unique_authors", "unique_musics",
    "finish_count", "like_count", "avg_duration_time",
    "finish_rate", "like_rate",
    "view_bin", "view_bin_order",
    "like_bin", "like_bin_order",
    "finish_bin", "finish_bin_order",
    "duration_bin", "duration_bin_order"
]

user_features_powerbi = user_df[powerbi_columns].copy()
user_features_powerbi.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"Exported {len(user_features_powerbi):,} users to {OUTPUT_FILE}")
user_features_powerbi.head()


Exported 59,232 users to ../data/processed/user_features_powerbi.csv


,uid,total_views,unique_videos,unique_authors,unique_musics,finish_count,like_count,avg_duration_time,finish_rate,like_rate,view_bin,view_bin_order,like_bin,like_bin_order,finish_bin,finish_bin_order,duration_bin,duration_bin_order
0,0,34,34,31,31,18,0,12.06,0.5294,0.0000,26–50,4,0 likes,1,11–20 finishes,4,12–15s,3
1,1,28,28,28,26,14,1,12.36,0.5000,0.0357,26–50,4,1 like,2,11–20 finishes,4,12–15s,3
2,2,56,56,56,47,19,0,10.36,0.3393,0.0000,51–100,5,0 likes,1,11–20 finishes,4,10–12s,2
3,3,117,117,116,89,60,1,9.98,0.5128,0.0085,101+,6,1 like,2,51+ finishes,7,≤10s,1
4,4,123,123,117,94,77,0,10.85,0.6260,0.0000,101+,6,0 likes,1,51+ finishes,7,10–12s,2
